# Spotify Artist Popularity Ranking — Standalone Analysis

**Research question:** How can cumulative streaming success and current daily listening activity be combined into an interpretable artist popularity ranking?

**Input:** `artists.csv` in the same directory. This analysis is independent of the Spotify 2023 song notebook.

**Important limitations:** The dataset is a ranked/top-artist sample, not all Spotify artists; the capture date is not established by the CSV. These are *snapshot-relative* rankings, not proof of song-level predictive ability. Figures are in **millions of streams**. Daily means **millions of streams per day** at capture. The weights below are a starting hypothesis, not a validated scientific scale.

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

path = Path("artists.csv")
if not path.exists():
    raise FileNotFoundError("Put artists.csv in the same directory as this notebook")
raw = pd.read_csv(path)
print("Shape:", raw.shape)
display(raw.head())

## 1. Clean and audit the artist data

`Streams`, `As lead`, `Solo`, and `As feature` often contain commas. **Do not sum** the role-based columns into total streams; `Solo` is a subset of lead and lead/feature represent overlapping categories of credit. We keep duplicated names visible rather than merging potentially different artists (e.g., two entries named “Eve”).

In [ ]:
df = raw.copy()
num_cols = ["Streams", "Daily", "As lead", "Solo", "As feature"]
for col in num_cols:
    df[col] = pd.to_numeric(
        df[col].astype("string").str.replace(",", "", regex=False),
        errors="coerce"
    )

print("Missing values:\n", df.isna().sum().to_string())
print("\nDuplicated artist names:\n", df.loc[df["Artist"].duplicated(keep=False), ["Artist", "Streams", "Daily"]].to_string(index=False))
print("\nSummary (millions):")
display(df[num_cols].describe())

## 2. Explore career size versus present activity

A high cumulative stream count need not imply a high current daily count. Because both variables are strongly skewed, plot them on logarithmic axes.

In [ ]:
plot_df = df.loc[(df["Streams"] > 0) & (df["Daily"] > 0)].copy()
fig, ax = plt.subplots(figsize=(8, 5))
ax.scatter(plot_df["Streams"], plot_df["Daily"], alpha=0.35, s=16)
ax.set(xscale="log", yscale="log", xlabel="Lifetime streams (millions)", ylabel="Daily streams (millions/day)", title="Artist career success vs. daily listening")
plt.show()
print("Spearman correlation:", round(plot_df[["Streams", "Daily"]].corr(method="spearman").iloc[0, 1], 3))

## 3. Build the Artist Popularity Score (APS)

1. Apply `log1p` to reduce the disproportionate influence of superstar counts.
2. Convert each transformed variable to a percentile from 0 to 100 *within the observed sample*.
3. Average lifetime and daily percentiles with equal weights.

\[
APS_j = 0.5\, P_{j,\mathrm{lifetime}} + 0.5\, P_{j,\mathrm{daily}}
\]

This is a transparent **composite ranking**, not an externally calibrated Spotify popularity score. Artists without a valid `Daily` value are omitted from the main combined ranking; they remain in the cleaned data.

In [ ]:
ranked = df.loc[(df["Streams"] >= 0) & (df["Daily"] >= 0)].copy()
ranked["log_lifetime"] = np.log1p(ranked["Streams"])
ranked["log_daily"] = np.log1p(ranked["Daily"])
ranked["career_percentile"] = ranked["log_lifetime"].rank(pct=True) * 100
ranked["daily_percentile"] = ranked["log_daily"].rank(pct=True) * 100

LIFETIME_WEIGHT = 0.50
ranked["APS"] = (LIFETIME_WEIGHT * ranked["career_percentile"]
                 + (1 - LIFETIME_WEIGHT) * ranked["daily_percentile"])
ranked = ranked.sort_values(["APS", "Streams"], ascending=False).copy()
ranked["APS_rank"] = ranked["APS"].rank(ascending=False, method="min").astype(int)
print(f"Ranked {len(ranked):,} artists; omitted {len(df)-len(ranked)} with unavailable daily/stream values")
display(ranked[["APS_rank", "Artist", "Streams", "Daily", "career_percentile", "daily_percentile", "APS"]].head(20).round(2))

## 4. Compare original lifetime ranking against composite popularity

Look for artists with large differences between current momentum and career accumulation. Differences are **descriptive**, not evidence of increasing or decreasing popularity over time (we have only one snapshot).

In [ ]:
ranked["career_rank"] = ranked["Streams"].rank(ascending=False, method="min")
ranked["daily_rank"] = ranked["Daily"].rank(ascending=False, method="min")
ranked["rank_shift"] = ranked["career_rank"] - ranked["APS_rank"]
print("Artists ranked much higher by combined score than lifetime streams:")
display(ranked.nlargest(12, "rank_shift")[["Artist", "career_rank", "daily_rank", "APS_rank", "rank_shift"]])
print("Artists ranked lower by combined score than lifetime streams:")
display(ranked.nsmallest(12, "rank_shift")[["Artist", "career_rank", "daily_rank", "APS_rank", "rank_shift"]])

In [ ]:
top = ranked.nsmallest(15, "APS_rank").sort_values("APS")
fig, ax = plt.subplots(figsize=(9, 6))
ax.barh(top["Artist"], top["APS"])
ax.set(xlabel="Artist Popularity Score (0–100, sample percentile composite)", title="Top 15 artists by Artist Popularity Score")
plt.tight_layout()
plt.show()

## 5. Sensitivity test: do weights change the rankings?

Recompute scores when the importance of historical versus daily streams changes. Because weights are judgment calls, stability across plausible settings is important.

In [ ]:
for lifetime_weight in [0.25, 0.50, 0.75]:
    col = f"APS_lifetime_{int(100*lifetime_weight)}"
    ranked[col] = (lifetime_weight * ranked["career_percentile"]
                   + (1-lifetime_weight) * ranked["daily_percentile"])
    print(f"\nTop 10 — lifetime weight {lifetime_weight:.0%}:")
    display(ranked.nlargest(10, col)[["Artist", col]].round(2))

## 6. Explore solo and featured contributions (secondary descriptive analysis)

These are **not independent components** of total streams. They can illuminate artist roles, but should not be added together as if they were new streams.

In [ ]:
roles = df.loc[df["Streams"].gt(0)].copy()
roles["feature_share"] = roles["As feature"] / roles["Streams"]
roles["solo_share"] = roles["Solo"] / roles["Streams"]
print("Role ratios (nonmissing rows):")
display(roles[["feature_share", "solo_share"]].describe().round(3))

## 7. Export results and document limitations

- `artist_popularity_rankings.csv`: all artists with both streaming statistics, sorted by APS.
- The file's **snapshot date** and ranking methodology should be verified from Kaggle/source documentation before writing conclusions.
- The top-3,000 sample is selective. Percentiles reflect **only this set of artists**.
- Name collisions mean `Artist` is not a globally unique ID.
- To later predict song popularity, acquire dated *pre-release* artist metrics and link with artist IDs; this single snapshot is not enough for leakage-free historical prediction.

In [ ]:
columns = ["APS_rank", "Artist", "APS", "Streams", "Daily", "career_percentile", "daily_percentile", "career_rank", "daily_rank", "rank_shift"]
ranked[columns].sort_values("APS_rank").to_csv("artist_popularity_rankings.csv", index=False)
print("Saved artist_popularity_rankings.csv")